In [ ]:
%%bash
# Cell 1: environment for the authors' 2021 code (Python 3.8, torch 1.13.1, bundled transformers 3.2.0)
set -e
nvidia-smi --query-gpu=name,memory.total --format=csv
cd /content
[ -d PrefixTuning ] || git clone -q -b cleaned https://github.com/XiangLi1999/PrefixTuning
if [ ! -x /content/py38/bin/python ]; then
  pip -q install uv
  uv venv /content/py38 --python 3.8
  uv pip install --python /content/py38/bin/python torch==1.13.1 nltk sentencepiece -e /content/PrefixTuning/transformers
fi
/content/py38/bin/python -c "import torch, transformers; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'transformers', transformers.__version__)"
python --version
# patch the hard-coded server paths in run_generation.py
cd /content/PrefixTuning/gpt2
[ -f run_generation_orig.py ] || cp run_generation.py run_generation_orig.py
sed -e 's#/u/scr/xlisali/e2e_data/#/content/PrefixTuning/data/e2e_data/#g' -e 's#/u/scr/xlisali/contrast_LM/transformers/examples/text-generation/#/content/gen_out/#g' -e 's#python /u/scr/xlisali/e2e-metrics/measure_scores.py#/usr/bin/python3 /content/e2e-metrics/measure_scores.py#g' run_generation_orig.py > run_generation.py
mkdir -p /content/gen_out/e2e_results
echo "patched lines: $(diff run_generation_orig.py run_generation.py | grep -c '^>')"


name, memory.total [MiB]
Tesla T4, 15360 MiB


Using CPython 3.8.20
Creating virtual environment at: py38
Activate with: source py38/bin/activate
Using Python 3.8.20 environment at: py38
Resolved 25 packages in 1.61s
Prepared 24 packages in 33.88s
Installed 25 packages in 189ms
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + click==8.1.8
 + filelock==3.16.1
 + idna==3.15
 + joblib==1.4.2
 + nltk==3.9.1
 + numpy==1.24.4
 + nvidia-cublas-cu11==11.10.3.66
 + nvidia-cuda-nvrtc-cu11==11.7.99
 + nvidia-cuda-runtime-cu11==11.7.99
 + nvidia-cudnn-cu11==8.5.0.96
 + packaging==26.2
 + regex==2024.11.6
 + requests==2.32.4
 + sacremoses==0.1.1
 + sentencepiece==0.2.0
 + setuptools==75.3.4
 + tokenizers==0.8.1rc2
 + torch==1.13.1
 + tqdm==4.70.1
 + transformers==3.2.0 (from file:///content/PrefixTuning/transformers)
 + typing-extensions==4.13.2
 + urllib3==2.2.3
 + wheel==0.45.1
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/content/PrefixTuning/transformers/src/transformers/__init__.py", line 124, in 

CalledProcessError: Command 'b'# Cell 1: environment for the authors\' 2021 code (Python 3.8, torch 1.13.1, bundled transformers 3.2.0)\nset -e\nnvidia-smi --query-gpu=name,memory.total --format=csv\ncd /content\n[ -d PrefixTuning ] || git clone -q -b cleaned https://github.com/XiangLi1999/PrefixTuning\nif [ ! -x /content/py38/bin/python ]; then\n  pip -q install uv\n  uv venv /content/py38 --python 3.8\n  uv pip install --python /content/py38/bin/python torch==1.13.1 nltk sentencepiece -e /content/PrefixTuning/transformers\nfi\n/content/py38/bin/python -c "import torch, transformers; print(\'torch\', torch.__version__, \'cuda\', torch.cuda.is_available(), \'transformers\', transformers.__version__)"\npython --version\n# patch the hard-coded server paths in run_generation.py\ncd /content/PrefixTuning/gpt2\n[ -f run_generation_orig.py ] || cp run_generation.py run_generation_orig.py\nsed -e \'s#/u/scr/xlisali/e2e_data/#/content/PrefixTuning/data/e2e_data/#g\' -e \'s#/u/scr/xlisali/contr

In [ ]:
# Cell 2a: finish the environment (the bundled transformers 3.2.0 also needs 'six') and patch the server paths
import os, subprocess
def sh(c):
    r = subprocess.run(c, shell=True, executable="/bin/bash", capture_output=True, text=True)
    print((r.stdout + r.stderr)[-2500:])
sh("uv pip install --python /content/py38/bin/python six sacremoses 2>&1 | tail -2")
sh("/content/py38/bin/python -c \"import torch, transformers; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'transformers', transformers.__version__)\"")
sh("""cd /content/PrefixTuning/gpt2
[ -f run_generation_orig.py ] || cp run_generation.py run_generation_orig.py
sed -e 's#/u/scr/xlisali/e2e_data/#/content/PrefixTuning/data/e2e_data/#g' -e 's#/u/scr/xlisali/contrast_LM/transformers/examples/text-generation/#/content/gen_out/#g' -e 's#python /u/scr/xlisali/e2e-metrics/measure_scores.py#/usr/bin/python3 /content/e2e-metrics/measure_scores.py#g' run_generation_orig.py > run_generation.py
mkdir -p /content/gen_out/e2e_results
echo patched lines: $(diff run_generation_orig.py run_generation.py | grep -c '^>')""")

# Cell 2: GPT-2 medium weights + training with the authors' code (1 epoch, prefix length 10, ~30 min on a T4)
import os, subprocess
from huggingface_hub import snapshot_download
if not os.path.exists("/content/gpt2-medium/pytorch_model.bin"):
    snapshot_download("gpt2-medium", local_dir="/content/gpt2-medium",
                      allow_patterns=["config.json", "vocab.json", "merges.txt", "pytorch_model.bin"])
print(os.listdir("/content/gpt2-medium"))
# NB: the folder name must NOT contain "prefixtune": run_language_modeling.py calls the buggy gen.py at the end
cmd = ("cd /content/PrefixTuning/gpt2 && /content/py38/bin/python run_language_modeling.py "
       "--output_dir=save_e2e_models_convcheck/e2e_p10_e1 --model_type=gpt2 "
       "--model_name_or_path=/content/gpt2-medium --tokenizer_name=/content/gpt2-medium "
       "--per_device_train_batch_size 5 --per_device_eval_batch_size 5 --save_steps 500000 --num_train_epochs 1 "
       "--do_train --train_data_file=../data/e2e_data/src1_train.txt --line_by_line --save_total_limit 1 --overwrite_output_dir "
       "--task_mode data2text --eval_data_file=../data/e2e_data/src1_valid.txt --tuning_mode prefixtune "
       "--logging_dir save_e2e_models_convcheck/runs/e2e_p10_e1 --train_embs no --optim_prefix yes --preseqlen 10 "
       "--prefix_mode activation --format_mode cat --gradient_accumulation_steps 1 --learning_rate 5e-05 --weight_decay 0.0 "
       "--seed 101 --disable_tqdm --mid_dim 512 --init_random no --use_dropout no --prefix_dropout 0.0 --objective_mode 1 "
       "--logging_steps 500 --cache_dir /content/cache > /content/train_p10_e1.log 2>&1")
r = subprocess.run(cmd, shell=True)
print("exit code", r.returncode)
print(subprocess.run("grep -E 'loss|Num examples|optimization steps|total param|preseqlen|Training completed|Error|error' /content/train_p10_e1.log | tail -40",
                     shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("ls -la /content/PrefixTuning/gpt2/save_e2e_models_convcheck/e2e_p10_e1", shell=True, capture_output=True, text=True).stdout)


Installed 1 package in 0.85ms
 + six==1.17.0

torch 1.13.1+cu117 cuda True transformers 3.2.0

patched lines: 18



Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

['config.json', '.cache', 'merges.txt', 'vocab.json', 'pytorch_model.bin']
exit code 0
11/02/2026 10:00:59 - INFO - __main__ -   Training/evaluation parameters TrainingArguments(output_dir='save_e2e_models_convcheck/e2e_p10_e1', overwrite_output_dir=True, do_train=True, do_eval=False, do_predict=False, evaluate_during_training=False, evaluation_strategy=<EvaluationStrategy.NO: 'no'>, prediction_loss_only=False, per_device_train_batch_size=5, per_device_eval_batch_size=5, per_gpu_train_batch_size=None, per_gpu_eval_batch_size=None, gradient_accumulation_steps=1, learning_rate=5e-05, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, max_grad_norm=1.0, num_train_epochs=1.0, max_steps=-1, warmup_steps=0, logging_dir='save_e2e_models_convcheck/runs/e2e_p10_e1', logging_first_step=False, logging_steps=500, save_steps=500000, save_total_limit=1, no_cuda=False, seed=101, fp16=False, fp16_opt_level='O1', local_rank=-1, tpu_num_cores=None, tpu_metrics_debug=False, debug=Fal

In [ ]:
# Cell 3: generate for the 630 E2E test MRs with the authors' run_generation.py (beam 5), then score
import os, re, json, subprocess
def sh(c):
    r = subprocess.run(c, shell=True, executable="/bin/bash", capture_output=True, text=True)
    print((r.stdout + r.stderr)[-3000:])
# run_generation.py ALSO reads the prefix format from the folder name (it must contain "cat"), so link the model folder under such a name
sh("cd /content/PrefixTuning/gpt2/save_e2e_models_convcheck && ln -sfn e2e_p10_e1 e2e_p10_e1_act_cat && ls -la")
MODEL = "save_e2e_models_convcheck/e2e_p10_e1_act_cat"
gen = ("cd /content/PrefixTuning/gpt2 && /content/py38/bin/python run_generation.py --model_type=gpt2 --length 100 "
       "--model_name_or_path=/content/gpt2-medium --tokenizer_name=" + MODEL + " --num_return_sequences 5 --stop_token [EOS] "
       "--task_mode=data2text --control_mode=yes --tuning_mode prefixtune --gen_dir e2e_results --eval_dataset test "
       "--optim_prefix yes --preseqlen 10 --prefix_mode activation --format_mode cat "
       "--prefixModel_name_or_path " + MODEL + " --cache_dir /content/cache > /content/gen_e2e_p10_e1.log 2>&1")
# generation of 630 MRs with beam 5 takes about 15 minutes
print("generation exit code", subprocess.run(gen, shell=True).returncode)
sh("tail -4 /content/gen_e2e_p10_e1.log; ls -la /content/gen_out/e2e_results/; wc -l /content/gen_out/e2e_results/*")

# --- score with the same NLTK / rouge_score scorer as my peft runs (src/train_prefix.py) ---
sh("pip -q install nltk rouge-score")
import nltk; nltk.download("wordnet", quiet=True)
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.nist_score import corpus_nist
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
D = "/content/gen_out/e2e_results/"
name = os.path.basename(MODEL)
def read_groups(p):
    g, cur = [], []
    for line in open(p, encoding="utf-8"):
        line = line.strip()
        if line: cur.append(line)
        elif cur: g.append(cur); cur = []
    if cur: g.append(cur)
    return g
refs = read_groups(D + name + "_test_gold")
hyps = [l.strip() for l in open(D + name + "_test_beam", encoding="utf-8") if l.strip()]
srcs = [l.strip() for l in open(D + name + "_test_src", encoding="utf-8") if l.strip()]
print(len(refs), "reference groups,", len(hyps), "outputs,", len(srcs), "inputs")
tok = lambda s: re.findall(r"\w+|[^\w\s]", s.lower())
H = [tok(h) for h in hyps]; R = [[tok(r) for r in rs] for rs in refs]
sc = rouge_scorer.RougeScorer(["rougeL"])
res = {"n": len(hyps), "BLEU": 100 * corpus_bleu(R, H), "NIST": corpus_nist(R, H, n=5),
       "METEOR": 100 * sum(meteor_score(r, h) for h, r in zip(H, R)) / len(H),
       "ROUGE_L": 100 * sum(max(sc.score(r, h)["rougeL"].fmeasure for r in rs) for h, rs in zip(hyps, refs)) / len(hyps)}
# slot coverage on the same six slots as my peft runs
keymap = {"name": "name", "type": "eattype", "food": "food", "price": "pricerange", "area": "area", "near": "near"}
per, f_all, t_all = {}, 0, 0
for s, h in zip(srcs, hyps):
    s = s.replace("<|endoftext|>", "")
    for part in s.split("|"):
        if ":" not in part: continue
        k, v = [x.strip() for x in part.split(":", 1)]
        k = keymap.get(k.lower())
        if not k: continue
        ok = v.lower() in h.lower()
        a, b = per.get(k, (0, 0)); per[k] = (a + ok, b + 1); f_all += ok; t_all += 1
res["slot_coverage_%"] = 100 * f_all / max(1, t_all)
res["coverage_by_slot"] = {k: f"{100*a/b:.0f}% ({a}/{b})" for k, (a, b) in per.items()}
print(json.dumps(res, indent=2))
json.dump(res, open("/content/authors_code_scores.json", "w"), indent=2)
for s, h in list(zip(srcs, hyps))[:8]: print("IN :", s, "\nOUT:", h)


total 12
drwxr-xr-x 3 root root 4096 Nov  2 10:37 .
drwxr-xr-x 4 root root 4096 Nov  2 10:01 ..
drwxr-xr-x 2 root root 4096 Nov  2 10:33 e2e_p10_e1
lrwxrwxrwx 1 root root   10 Nov  2 10:37 e2e_p10_e1_act_cat -> e2e_p10_e1

generation exit code 0

/content/gen_out/e2e_results/e2e_p10_e1_act_cat_test_beam_eval 
 /content/gen_out/e2e_results/e2e_p10_e1_act_cat_test_gold 
 /content/gen_out/e2e_results/e2e_p10_e1_act_cat_test_beam
total 1580
drwxr-xr-x 2 root root   4096 Nov  2 10:49 .
drwxr-xr-x 3 root root   4096 Nov  2 10:00 ..
-rw-r--r-- 1 root root  85904 Nov  2 10:49 e2e_p10_e1_act_cat_test_beam
-rw-r--r-- 1 root root      0 Nov  2 10:49 e2e_p10_e1_act_cat_test_beam_eval
-rw-r--r-- 1 root root 657663 Nov  2 10:37 e2e_p10_e1_act_cat_test_gold
-rw-r--r-- 1 root root 101273 Nov  2 10:37 e2e_p10_e1_act_cat_test_src
-rw-r--r-- 1 root root      0 Nov  2 10:33 e2e_p10_e1_test_beam
-rw-r--r-- 1 root root 657663 Nov  2 10:33 e2e_p10_e1_test_gold
-rw-r--r-- 1 root root 101273 Nov  2 10:33 e2e_p

In [ ]:
# Cell 4: the official E2E scorer (as used in the paper). Java is installed; Colab already has numpy/matplotlib/scikit-image.
import subprocess
def sh(c, t=900):
    try:
        r = subprocess.run(c, shell=True, executable="/bin/bash", capture_output=True, text=True, timeout=t)
        print((r.stdout + r.stderr)[-4000:])
    except subprocess.TimeoutExpired:
        print("TIMEOUT after", t, "s:", c[:80])
sh("timeout 120 pip -q install future 2>&1 | tail -2; java -version 2>&1 | grep -i version")
D = "/content/gen_out/e2e_results/e2e_p10_e1_act_cat_test_"
sh(f"cd /content/e2e-metrics && timeout 840 python measure_scores.py {D}gold {D}beam 2>&1 | grep -v -i warning | tail -12")


openjdk version "21.0.12.1" 2026-08-18



KeyboardInterrupt: 